# 01 - Train the latent DiT scaling runs

    This is the main training notebook. 

    - DiT-Tiny/8 for 10k steps
    - DiT-Tiny/4 for 10k steps
    - DiT-Tiny/2 for 10k steps
    - DiT-S/4 for 20k steps


## Imports and device setup

    This cell imports the project modules and creates paths for checkpoints, logs, samples, and figures.


In [ ]:
import sys, time, math, json
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dit.configs import make_dit_config, TrainConfig, RUNS
from dit.models import make_model, count_parameters
from dit.diffusion import GaussianDiffusion
from dit.ema import EMA
from dit.data import LatentTensorDataset
from dit.latent import load_vae
from dit.train_utils import set_seed, make_loader, save_checkpoint, load_checkpoint, append_csv, validation_loss, save_json
from dit.sample_utils import save_class_grid

device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))

DATA_DIR = PROJECT_ROOT / "data"
LATENT_DIR = DATA_DIR / "imagenette_latents"
CKPT_ROOT = PROJECT_ROOT / "checkpoints" / "latent_imagenette"
RUNS_ROOT = PROJECT_ROOT / "runs" / "latent_imagenette"
SAMPLE_ROOT = PROJECT_ROOT / "samples" / "latent_imagenette"
FIG_DIR = PROJECT_ROOT / "figures"
for d in [CKPT_ROOT, RUNS_ROOT, SAMPLE_ROOT, FIG_DIR]: d.mkdir(parents=True, exist_ok=True)


## Configure the  plan

    This cell defines the actual training plan. 


In [ ]:
RUN_PROFILE = "all_runs"  # options: "all_runs" or "debug"
RESUME = True
TRAIN_ALL_RUNS = True
SELECTED_RUN_IDS = None  # e.g., ["DiT-Tiny-4", "DiT-S-4"] if TRAIN_ALL_RUNS=False

if RUN_PROFILE == "debug":
    planned_runs = [{**r, "target_steps": 500} for r in RUNS]
    print("WARNING: debug profile trains only 500 steps and is not a meaningful result.")
else:
    planned_runs = RUNS

if not TRAIN_ALL_RUNS and SELECTED_RUN_IDS is not None:
    planned_runs = [r for r in planned_runs if f"{r['model_name']}-{r['patch_size']}" in SELECTED_RUN_IDS]

pd.DataFrame(planned_runs)


## Load cached latent datasets

    This cell loads the VAE latents created by `00_setup_and_data.ipynb`. The model trains directly on these 32x32x4 tensors.


In [ ]:
train_latents = LatentTensorDataset(LATENT_DIR / "train")
val_latents = LatentTensorDataset(LATENT_DIR / "val")
print("Train latents:", len(train_latents), train_latents.latents.shape)
print("Val latents:", len(val_latents), val_latents.latents.shape)
if len(train_latents) < 5000:
    print("WARNING: fewer than 5000 training latents; results may be weak.")


## Load the VAE decoder for progress samples

    The training loop periodically samples latents from the DiT and decodes them into images, so we load the frozen VAE decoder here.


In [ ]:
vae = load_vae(device)


## Define the training function

    This cell defines a reusable function that trains one configuration until its target step. It saves raw and EMA weights, validation losses, progress samples, and history CSVs.


In [ ]:
def train_one_run(run_spec):
    run_id = f"{run_spec['model_name']}-{run_spec['patch_size']}"
    target_steps = int(run_spec["target_steps"])
    print(f"===== Training {run_id} to {target_steps} steps =====")
    if target_steps < 10_000 and RUN_PROFILE != "debug":
        print("WARNING: target_steps < 10k; this is unlikely to generate meaningful images.")

    dit_cfg = make_dit_config(run_spec["model_name"], patch_size=int(run_spec["patch_size"]))
    # Conservative batch choices for a single-GPU workflow.
    if dit_cfg.model_name == "DiT-S":
        batch_size, grad_accum = 16, 4
    elif dit_cfg.patch_size == 2:
        batch_size, grad_accum = 16, 4
    else:
        batch_size, grad_accum = 32, 2
    train_cfg = TrainConfig(
        run_mode=RUN_PROFILE,
        train_steps=target_steps,
        batch_size=batch_size,
        grad_accum_steps=grad_accum,
        ema_decay=0.999,
        val_every=500 if RUN_PROFILE == "debug" else 1000,
        sample_every=500 if RUN_PROFILE == "debug" else 2000,
        save_every=500 if RUN_PROFILE == "debug" else 2000,
        num_workers=4,
        mixed_precision=True,
        seed=42,
    )
    set_seed(train_cfg.seed)
    out_dir = CKPT_ROOT / run_id
    run_dir = RUNS_ROOT / run_id
    sample_dir = SAMPLE_ROOT / run_id / "train_progress"
    out_dir.mkdir(parents=True, exist_ok=True)
    run_dir.mkdir(parents=True, exist_ok=True)
    sample_dir.mkdir(parents=True, exist_ok=True)
    save_json(run_dir / "dit_config.json", dit_cfg.to_dict())
    save_json(run_dir / "train_config.json", train_cfg.to_dict())

    model = make_model(dit_cfg).to(device)
    diffusion = GaussianDiffusion(train_cfg.diffusion_steps, device=device)
    ema = EMA(model, decay=train_cfg.ema_decay)
    optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg.learning_rate, weight_decay=train_cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda" and train_cfg.mixed_precision))
    train_loader = make_loader(train_latents, train_cfg.batch_size, shuffle=True, num_workers=train_cfg.num_workers, seed=train_cfg.seed)
    val_loader = make_loader(val_latents, train_cfg.batch_size, shuffle=False, num_workers=train_cfg.num_workers, seed=train_cfg.seed)
    train_iter = iter(train_loader)
    start_step = 0
    latest = out_dir / "latest.pt"
    if RESUME and latest.exists():
        ckpt = load_checkpoint(latest, model, optimizer, scaler, ema, device=device)
        start_step = int(ckpt.get("step", 0))
        print("Resumed from", latest, "at step", start_step)
    if start_step >= target_steps:
        print(f"{run_id} already reached {start_step} steps; skipping training.")
        return {"run_id": run_id, "status": "already_complete", "step": start_step}

    print("Parameters:", count_parameters(model), "Tokens:", dit_cfg.num_tokens, "Effective batch:", train_cfg.batch_size * train_cfg.grad_accum_steps)
    model.train()
    accum_loss = 0.0
    tic = time.time()
    pbar = tqdm(range(start_step + 1, target_steps + 1), desc=run_id)
    optimizer.zero_grad(set_to_none=True)
    for step in pbar:
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)
        x = x.to(device, non_blocking=True).float()
        y = y.to(device, non_blocking=True).long()
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda" and train_cfg.mixed_precision)):
            loss = diffusion.training_losses(model, x, y)["loss"] / train_cfg.grad_accum_steps
        scaler.scale(loss).backward()
        accum_loss += float(loss.item()) * train_cfg.grad_accum_steps
        if step % train_cfg.grad_accum_steps == 0:
            if train_cfg.grad_clip:
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
            else:
                grad_norm = torch.tensor(float("nan"))
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            ema.update(model)
        if step % train_cfg.log_every == 0:
            seconds_per_step = (time.time() - tic) / train_cfg.log_every
            row = {
                "step": step,
                "train_loss": accum_loss / train_cfg.log_every,
                "val_loss": float("nan"),
                "lr": optimizer.param_groups[0]["lr"],
                "grad_norm": float(grad_norm.item()) if 'grad_norm' in locals() else float("nan"),
                "seconds_per_step": seconds_per_step,
                "ema_decay": train_cfg.ema_decay,
                "run_mode": RUN_PROFILE,
            }
            append_csv(run_dir / "history.csv", row)
            pbar.set_postfix(loss=row["train_loss"], sec=seconds_per_step)
            accum_loss = 0.0
            tic = time.time()
        if step % train_cfg.val_every == 0:
            vals = validation_loss(model, diffusion, val_loader, device, max_batches=20 if RUN_PROFILE == "debug" else 50)
            row = {
                "step": step,
                "train_loss": float("nan"),
                **vals,
                "lr": optimizer.param_groups[0]["lr"],
                "grad_norm": float("nan"),
                "seconds_per_step": float("nan"),
                "ema_decay": train_cfg.ema_decay,
                "run_mode": RUN_PROFILE,
            }
            append_csv(run_dir / "history.csv", row)
            print("Validation:", vals)
        if step % train_cfg.sample_every == 0:
            ema.apply_to(model)
            grid_path = sample_dir / f"step_{step:07d}_ema.png"
            save_class_grid(model, diffusion, vae, grid_path, num_classes=10, samples_per_class=2, ddim_steps=50, cfg_scale=1.5, device=device, progress=False)
            ema.restore(model)
            if step == train_cfg.sample_every:
                raw_path = sample_dir / f"step_{step:07d}_raw.png"
                save_class_grid(model, diffusion, vae, raw_path, num_classes=10, samples_per_class=2, ddim_steps=50, cfg_scale=1.5, device=device, progress=False)
            print("Saved progress sample:", grid_path)
        if step % train_cfg.save_every == 0 or step == target_steps:
            ckpt_path = out_dir / f"step_{step:07d}.pt"
            save_checkpoint(ckpt_path, model, optimizer, scaler, ema, step, dit_cfg.to_dict(), train_cfg.to_dict(), extra={"run_spec": run_spec})
            save_checkpoint(latest, model, optimizer, scaler, ema, step, dit_cfg.to_dict(), train_cfg.to_dict(), extra={"run_spec": run_spec})
            print("Saved checkpoint:", ckpt_path)
    return {"run_id": run_id, "status": "ok", "step": target_steps}


## Train the planned configurations

    This cell performs the actual training runs. It may take many hours depending on the GPU.


In [ ]:
results = []
for run_spec in planned_runs:
    results.append(train_one_run(run_spec))
pd.DataFrame(results)


## Visualize training curves

    This cell loads each run's history CSV and plots training/validation loss.


In [ ]:
import pandas as pd
plt.figure(figsize=(10, 6))
for run_spec in planned_runs:
    run_id = f"{run_spec['model_name']}-{run_spec['patch_size']}"
    hist_path = RUNS_ROOT / run_id / "history.csv"
    if not hist_path.exists():
        continue
    hist = pd.read_csv(hist_path)
    train_rows = hist[hist["train_loss"].notna()]
    val_rows = hist[hist["val_loss"].notna()]
    if len(train_rows):
        plt.plot(train_rows["step"], train_rows["train_loss"], alpha=0.4, label=f"{run_id} train")
    if len(val_rows):
        plt.plot(val_rows["step"], val_rows["val_loss"], linewidth=2, label=f"{run_id} val")
plt.xlabel("Training step")
plt.ylabel("Noise prediction MSE")
plt.title("DiT training curves")
plt.legend()
plt.grid(True, alpha=0.3)
out = FIG_DIR / "training_curves.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", out)
